# Testing the standalone multitask NLP service (Docker)

This notebook drives the `nlp-services-standalone` container — the Dockerized version of
`build_multitask_pipeline.ipynb`. You send a payload, poll a session, and get back flat JSON
annotation rows.

**Start the container first** (from `nlp-services-standalone/`):
```bash
export SPARK_NLP_SECRET=$(python3 -c "import json;print(json.load(open('/home/ubuntu/cabir/keys/6.4.1.spark_nlp_for_healthcare.json'))['SECRET'])")
docker compose -f dc.nlp-services-standalone.yaml up --build -d
```

**API** (async: submit → poll → fetch):

| Method | Path | |
|---|---|---|
| GET  | `/health` | status, `model_loaded`, queue depth, ES reachability |
| GET  | `/api/v1/model` | the one model this container serves + defaults |
| POST | `/api/v1/runs` | → `202` + `session_id` |
| GET  | `/api/v1/status/{session_id}` | poll until `completed` |
| GET  | `/api/v1/results/{session_id}` | `?limit=&offset=` |
| GET  | `/api/v1/logs/{session_id}` | `?tail=&full=` |

In [1]:
import json
import time

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 60)

BASE_URL = "http://localhost:8510"   # 5001 is the original custom_nlp_service; this one is 5002


def submit_run(payload):
    """POST /api/v1/runs -> 202 + session envelope (status_url / results_url / logs_url)."""
    resp = requests.post(f"{BASE_URL}/api/v1/runs", json=payload)
    resp.raise_for_status()
    return resp.json()


def wait_for(session_id, poll=3, timeout=900):
    """Poll /api/v1/status until the session reaches a terminal state."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        status = requests.get(f"{BASE_URL}/api/v1/status/{session_id}").json()
        print(f"  status={status['status']:22} progress={status['progress']:>3}%  "
              f"stage={status.get('current_stage')}")
        if status["status"] in ("completed", "completed_with_errors", "failed"):
            return status
        time.sleep(poll)
    raise TimeoutError(f"session {session_id} did not finish in {timeout}s")


def get_results(session_id, limit=5000, offset=0):
    """GET /api/v1/results/{session_id}."""
    resp = requests.get(f"{BASE_URL}/api/v1/results/{session_id}",
                        params={"limit": limit, "offset": offset})
    resp.raise_for_status()
    return resp.json()


def run(payload):
    """Full flow: submit -> poll -> fetch -> DataFrame. Returns (session_id, status, df)."""
    session = submit_run(payload)
    session_id = session["session_id"]
    print("session_id:", session_id)
    status = wait_for(session_id)
    if status["status"] == "failed":
        print("FAILED:", status.get("message"), status.get("errors"))
        return session_id, status, pd.DataFrame()
    body = get_results(session_id)
    df = pd.DataFrame(body["results"])
    if not df.empty:
        df["task_type"] = df["raw_metadata"].apply(lambda m: m.get("task_type"))
    print(f"\n{body['total']} result rows")
    return session_id, status, df

## 0. Health & model info

`/health` is `warming` while the model loads (~1–2 min on a cold start) and `ok` once ready.
`/api/v1/model` tells you which single model this container serves — a request for any other
`model_id` is rejected (see the errors section).

In [2]:
print("HEALTH")
print(json.dumps(requests.get(f"{BASE_URL}/health").json(), indent=2))

print("\nMODEL")
print(json.dumps(requests.get(f"{BASE_URL}/api/v1/model").json(), indent=2))

HEALTH
{
  "status": "ok",
  "service": "nlp-services-standalone",
  "model_id": "zeroshot_multitask_base",
  "model_loaded": true,
  "spark_initialized": true,
  "queue": {
    "worker_alive": true,
    "depth": 0
  },
  "dependencies": {
    "elasticsearch": "down"
  }
}

MODEL
{
  "model_id": "zeroshot_multitask_base",
  "engine": "jsl_zero_shot_multitask",
  "model_loaded": true,
  "tasks": [
    "ner",
    "structure",
    "classification",
    "relation"
  ],
  "supported_thresholds": [
    "entity_threshold",
    "structure_threshold",
    "classification_threshold",
    "relation_threshold"
  ],
  "defaults": {
    "entity_threshold": 0.6,
    "structure_threshold": 0.6,
    "classification_threshold": 0.6,
    "relation_threshold": 0.6
  }
}


## 1. Inline documents — all four tasks at once

The richest payload: send the text inline under `documents`, and a `zero_shot` block describing
**entities** (NER), **structures** (JSON extraction, with an enum `choices` field), **classifications**,
and **relations**. Each task also takes an optional threshold (default 0.6).

Relations are written the natural way (`"MEDICATION treats PROBLEM"`); the service underscore-joins
them for the annotator.

In [3]:
payload_all = {
    "documents": [
        {"document_id": "doc-0",
         "text": "Progress Note: Jennifer Smith is a 58-year-old woman with type 2 diabetes mellitus "
                 "and hypertension. She was started on metformin 500mg oral twice daily to treat her "
                 "diabetes. An HbA1c test was ordered to diagnose poor glycemic control."},
        {"document_id": "doc-1",
         "text": "The patient underwent an appendectomy for acute appendicitis. Postoperatively he "
                 "received ibuprofen 400mg tablet orally every 6 hours for pain."},
    ],
    "zero_shot": {
        "model_id": "zeroshot_multitask_base",           # optional; omit to use the pinned model
        "entities": [
            {"label": "PROBLEM",    "dtype": "str", "description": "A medical condition, symptom, or diagnosis"},
            {"label": "MEDICATION", "dtype": "str", "description": "Drug or pharmaceutical treatment"},
            {"label": "PROCEDURE",  "dtype": "str", "description": "Medical or surgical procedure"},
            {"label": "TEST",       "dtype": "str", "description": "Diagnostic test or lab result"},
        ],
        "structures": [
            {"name": "medication_item", "fields": [
                {"name": "drug_name", "dtype": "str", "description": "Name of the drug"},
                {"name": "dosage",    "dtype": "str", "description": "Dose amount and unit"},
                {"name": "frequency", "dtype": "str", "description": "How often taken"},
                {"name": "route",     "choices": ["oral", "IV", "topical", "subcutaneous"]},   # enum field
            ]},
        ],
        "classifications": [
            {"task": "document_type", "labels": ["Radiology Report", "Discharge Summary", "Progress Note"]},
        ],
        "relations": ["MEDICATION treats PROBLEM", "TEST diagnoses PROBLEM"],
        "entity_threshold": 0.6,
        "structure_threshold": 0.6,
        "classification_threshold": 0.6,
        "relation_threshold": 0.6,
    },
    "job_details": json.dumps({"source": "notebook", "mode": "zero_shot_multitask"}),
}

session_id, status, df = run(payload_all)
df["task_type"].value_counts()

session_id: dd616431-b63d-4ca4-82b3-0814dcd0bc82
  status=processing             progress= 30%  stage=extract_annotations
  status=processing             progress= 30%  stage=extract_annotations
  status=processing             progress= 30%  stage=extract_annotations
  status=completed              progress=100%  stage=persist_results

27 result rows


task_type
ner               10
relation           6
structure          6
classification     5
Name: count, dtype: int64

### The result rows

One row per annotation. `label` means something different per task, so read `raw_metadata.task_type`:

| task_type | rows | `label` | `span_text` |
|---|---|---|---|
| `ner` | 1 | entity type | the chunk |
| `classification` | 1 | task name | predicted label |
| `relation` | 2 (head + tail) | relation name | `chunk1` / `chunk2` |
| `structure` | 1 per field | field name | field text |

`sentence` is the full sentence text; its index is `raw_metadata.sentence_number`.

In [4]:
df["sentence_number"] = df["raw_metadata"].apply(lambda m: m.get("sentence_number"))
df[["document_id", "task_type", "label", "span_text", "start", "end",
    "sentence_number", "confidence", "engine"]]

,document_id,task_type,label,span_text,start,end,sentence_number,confidence,engine
0,doc-0,ner,PROBLEM,type 2 diabetes mellitus,58,81,0,0.967561,jsl_zero_shot_multitask
1,doc-0,classification,document_type,Progress Note,0,99,0,1.000000,jsl_zero_shot_multitask
2,doc-0,ner,MEDICATION,metformin,120,128,1,0.993419,jsl_zero_shot_multitask
3,doc-0,ner,PROBLEM,diabetes,166,173,1,0.997835,jsl_zero_shot_multitask
4,doc-0,classification,document_type,Progress Note,101,174,1,0.997836,jsl_zero_shot_multitask
5,doc-0,relation,MEDICATION_treats_PROBLEM,metformin,19,27,1,0.992592,jsl_zero_shot_multitask
6,doc-0,relation,MEDICATION_treats_PROBLEM,diabetes,65,72,1,0.999986,jsl_zero_shot_multitask
7,doc-0,structure,drug_name,metformin,19,28,1,0.999446,jsl_zero_shot_multitask
8,doc-0,structure,dosage,500mg,29,34,1,0.999998,jsl_zero_shot_multitask
9,doc-0,structure,frequency,twice daily,40,51,1,1.000000,jsl_zero_shot_multitask


In [5]:
# A single full row, so you can see every field the service returns.
print(json.dumps(df.iloc[0].drop("task_type").drop("sentence_number").to_dict(), indent=2, default=str))

{
  "row_id": "inline-0",
  "document_id": "doc-0",
  "patient_id": null,
  "visit_id": null,
  "job_id": null,
  "batch_id": null,
  "pipeline_id": "zeroshot_multitask_base",
  "engine": "jsl_zero_shot_multitask",
  "label": "PROBLEM",
  "span_text": "type 2 diabetes mellitus",
  "start": 58,
  "end": 81,
  "sentence": "Progress Note: Jennifer Smith is a 58-year-old woman with type 2 diabetes mellitus and hypertension.",
  "confidence": 0.9675615,
  "source_index": "inline",
  "source_mode": "custom_nlp_service",
  "raw_metadata": {
    "task_type": "ner",
    "ner_source": "extractions",
    "entity": "PROBLEM",
    "confidence": "0.9675615",
    "sentence_number": 0,
    "model_type": "multitask"
  },
  "created_at": "2026-07-30T21:28:45.384172+00:00",
  "session_id": "dd616431-b63d-4ca4-82b3-0814dcd0bc82",
  "job_details": "{\"source\": \"notebook\", \"mode\": \"zero_shot_multitask\"}",
  "result_id": "128b7d35facd8ffd5c093447c242ab53858be367"
}


## 2. Payload variants — entities can be bare strings, and aliases are accepted

Entities may be plain label strings (no description), and the service accepts `selected_labels` /
`supported_labels` as aliases for `entities` — so payloads written for the older service still work.
You only need one of entities / structures / classifications / relations.

In [17]:
TEXT = ("He was given boluses of MS04, he is on 80mg of oxycontin at home, and has also "
        "received ativan for anxiety. Prostate gland measures 10x1.1x4.9 cm and is mildly enlarged.")

# bare label strings, using the `supported_labels` alias
payload_bare = {
    "documents": [TEXT],   # a bare string is accepted too (becomes {"text": ...})
    "zero_shot": {"supported_labels": ["PROBLEM", "MEDICATION", "TEST", "PROCEDURE"]},
}
_, _, df_bare = run(payload_bare)
df_bare[["task_type", "label", "span_text", "start", "end", "confidence"]]

session_id: 0d66325a-74d4-40f4-b367-e3f2af96714b
  status=processing             progress= 30%  stage=extract_annotations
  status=completed              progress=100%  stage=persist_results

5 result rows


,task_type,label,span_text,start,end,confidence
0,ner,PROCEDURE,boluses,13,19,0.890796
1,ner,TEST,MS04,24,27,0.916094
2,ner,MEDICATION,oxycontin,47,55,0.999926
3,ner,MEDICATION,ativan,88,93,0.999987
4,ner,PROBLEM,anxiety,99,105,0.999837


## 3. Elasticsearch documents — `document_ids`

Instead of inline text, reference documents already in the `raw_extractions` index. The service
fetches them (needs `ES_ENABLED=true`, which the compose file sets). Rows come back with the real
`document_id` / `patient_id` / `row_id` and `source_index=raw_extractions`.

> If ES is disabled, this returns `503`; inline `documents` always work regardless.

In [6]:
payload_es = {
    "document_ids": ["10082662-RR-37"],
    "zero_shot": {
        "entities": [
            {"label": "PROBLEM", "dtype": "str", "description": "A medical condition or diagnosis"},
            {"label": "TEST",    "dtype": "str", "description": "Diagnostic test or imaging study"},
        ],
        "classifications": [],# [{"task": "document_type", "labels": ["Radiology Report", "Progress Note"]}],
        "relations": [] #["TEST diagnoses PROBLEM"],
    },
    "job_details": json.dumps({"source": "notebook", "mode": "zero_shot_multitask"}),
}

session_id_es, status_es, df_es = run(payload_es)
print("documents_loaded:", status_es.get("documents_loaded"), "| warnings:", status_es.get("warnings"))
if not df_es.empty:
    display(df_es[["document_id", "patient_id", "source_index", "task_type",
                   "label", "span_text", "confidence"]].head(15))

session_id: 75042ed2-eb18-43a9-a346-67c157f58044
  status=processing             progress=  5%  stage=load_documents
  status=failed                 progress=100%  stage=load_documents
FAILED: Session failed: Connection error caused by: ConnectionError(Connection error caused by: NameResolutionError(HTTPConnection(host='pj-nosql', port=9200): Failed to resolve 'pj-nosql' ([Errno -3] Temporary failure in name resolution))) ["Connection error caused by: ConnectionError(Connection error caused by: NameResolutionError(HTTPConnection(host='pj-nosql', port=9200): Failed to resolve 'pj-nosql' ([Errno -3] Temporary failure in name resolution)))"]
documents_loaded: 0 | warnings: []


## 4. Session logs

`/api/v1/logs/{session_id}` returns the plain-text log for a session (Spark/JSL chatter included).

In [7]:
logs = requests.get(f"{BASE_URL}/api/v1/logs/{session_id}", params={"tail": 15}).text
print(logs)

2026-07-30 21:28:45,383 - nlp_multitask - INFO - Starting session dd616431-b63d-4ca4-82b3-0814dcd0bc82
2026-07-30 21:28:45,383 - nlp_multitask - INFO - Session dd616431-b63d-4ca4-82b3-0814dcd0bc82 stage=load_documents stage_status=processing progress=5 message=Loading documents.
2026-07-30 21:28:45,383 - nlp_multitask - INFO - Session dd616431-b63d-4ca4-82b3-0814dcd0bc82 stage=load_documents stage_status=completed progress=25 message=Loaded 2 document(s).
2026-07-30 21:28:45,384 - nlp_multitask - INFO - Session dd616431-b63d-4ca4-82b3-0814dcd0bc82 stage=extract_annotations stage_status=processing progress=30 message=Running zeroshot_multitask_base over 2 document(s).
2026-07-30 21:28:54,144 - nlp_multitask - INFO - Session dd616431-b63d-4ca4-82b3-0814dcd0bc82 stage=extract_annotations stage_status=completed progress=75 message=Extracted 27 annotation row(s).
2026-07-30 21:28:54,144 - nlp_multitask - INFO - Session dd616431-b63d-4ca4-82b3-0814dcd0bc82 stage=persist_results stage_status=

## 5. Error handling

The service validates the request up front and returns `422` (or `503`) instead of queuing a run
that would fail:
- a `model_id` this container isn't pinned to → `422` (one model per container)
- an empty `zero_shot` (no task) → `422`
- neither `documents` nor `document_ids` → `422`

In [8]:
def show_error(label, payload):
    resp = requests.post(f"{BASE_URL}/api/v1/runs", json=payload)
    detail = resp.json().get("detail")
    msg = detail if isinstance(detail, str) else (detail[0]["msg"] if detail else "")
    print(f"{label:24} -> {resp.status_code}")
    print(f"   {str(msg)[:110]}\n")


show_error("wrong model_id", {
    "documents": ["some text"],
    "zero_shot": {"model_id": "zeroshot_multitask_generic", "entities": ["PROBLEM"]},
})
show_error("empty zero_shot", {"documents": ["some text"], "zero_shot": {}})
show_error("no document source", {"zero_shot": {"entities": ["PROBLEM"]}})

wrong model_id           -> 422
   This container serves 'zeroshot_multitask_base'. Model 'zeroshot_multitask_generic' cannot be served: Pretrain

empty zero_shot          -> 422
   Value error, zero_shot must define at least one task: entities, structures, classifications, or relations.

no document source       -> 422
   Value error, Provide 'documents' (inline text) and/or 'document_ids' (fetched from Elasticsearch).

